用 Python 随机生成 1,000 条均值为 0 的噪声时间序列，试搜索 50 条不同窗口的 SMA 交叉策略。

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

# ---------- 参数 ----------
N_SERIES   = 1000     # 噪声时间序列条数
LENGTH     = 252      # 每条序列长度（交易日；可改 1000 等）
N_STRATS   = 50       # 交叉策略数量
SEED       = 42       # 随机种子，保证结果可复现
ALPHA      = 0.05     # 显著性阈值
ANNUALIZE  = 252      # 年化因子；若 LENGTH=1000 可改成 1000

rng = np.random.default_rng(SEED)

# ---------- 1) 生成噪声价格序列 ----------
def generate_noise_prices(length, mu=0, sigma=1, start_price=100):
    """给定长度生成一条随机游走价格序列（均值为0的正态收益累积）"""
    returns = rng.normal(mu, sigma, size=length)
    prices  = start_price + np.cumsum(returns)
    return pd.Series(prices)

price_series_list = [
    generate_noise_prices(LENGTH) for _ in range(N_SERIES)
]

# ---------- 2) 随机抽取 50 组 SMA 窗口 ----------
#   short ∈ [3,60]，long ∈ [short+1, 200]；保证两两不同
windows = set()
while len(windows) < N_STRATS:
    s = rng.integers(3, 60)
    l = rng.integers(s + 1, 200)
    windows.add((s, l))
windows = list(windows)

# ---------- 3) 回测函数 ----------
def backtest_sma(price: pd.Series, short: int, long: int):
    """返回策略日收益、夏普、p-value"""
    sma_s = price.rolling(short).mean()
    sma_l = price.rolling(long).mean()

    # 交易信号：short SMA 上穿 long SMA 做多，否则空仓
    pos    = (sma_s > sma_l).astype(int)
    # 避免未来函数：实际持仓用前一天信号
    pos    = pos.shift(1).fillna(0)

    daily_ret = pos * price.pct_change().fillna(0)

    if daily_ret.std(ddof=1) == 0:
        return np.nan, np.nan

    sharpe = daily_ret.mean() / daily_ret.std(ddof=1) * np.sqrt(ANNUALIZE)
    t_stat = sharpe * np.sqrt(len(daily_ret))
    p_val  = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=len(daily_ret) - 1))
    return sharpe, p_val

# ---------- 4) 执行回测 ----------
records = []
for sid, price in enumerate(price_series_list):
    for wid, (s, l) in enumerate(windows):
        sharpe, p_val = backtest_sma(price, s, l)
        records.append({
            "series_id" : sid,
            "strategy_id": wid,
            "short_win" : s,
            "long_win"  : l,
            "sharpe"    : sharpe,
            "p_value"   : p_val,
            "significant": p_val < ALPHA if not np.isnan(p_val) else False
        })

results = pd.DataFrame(records)
results.to_csv("sma_backtest_results.csv", index=False, encoding="utf-8")

# ---------- 5) 统计显著性 ----------
total_tests     = len(results)
sig_tests       = results["significant"].sum()
sig_pct         = sig_tests / total_tests * 100
sig_per_series  = results.groupby("series_id")["significant"].sum()

print(f"总共回测策略数：{total_tests:,}")
print(f"显著 (p < {ALPHA}) 的策略数：{sig_tests:,}  "
      f"({sig_pct:.2f}%)")
print("每条时间序列平均出现显著策略数："
      f"{sig_per_series.mean():.2f}")

# 如果想查看详情，取消下一行注释
# results.head(20)


总共回测策略数：50,000
显著 (p < 0.05) 的策略数：41,529  (83.06%)
每条时间序列平均出现显著策略数：41.53


如果你要把“动量”因子和 RSI 指标结合，设计一个两层过滤的策略框架并说明信号流程。

In [2]:
def update_signals(price_df):
    # 更新周期：每天盘后
    today = price_df.index[-1]
    
    ## -------- 1. 动量层 --------
    monthly_close = price_df.resample("M").last()
    mom = monthly_close.pct_change(6).iloc[-1]          # 6 个月动量
    mom_rank = mom.rank(pct=True)
    
    long_candidates = mom_rank[mom_rank >= 0.70].index  # 强动量池
    short_candidates = mom_rank[mom_rank <= 0.30].index # 弱动量池
    
    ## -------- 2. RSI 层 --------
    rsi = talib.RSI(price_df["close"], timeperiod=14).iloc[-1]
    
    signals = {}
    for ticker in long_candidates:
        if rsi[ticker] < 30:
            signals[ticker] = {"action": "long", "size": 1}
        elif rsi[ticker] > 60:
            signals[ticker] = {"action": "flat"}
    
    for ticker in short_candidates:
        if rsi[ticker] > 70:
            signals[ticker] = {"action": "short", "size": 1}
        elif rsi[ticker] < 40:
            signals[ticker] = {"action": "flat"}
    
    return signals


时间序列回归（CAPM 或 FF3）来估 β

In [29]:
# -*- coding: utf-8 -*-
"""
Simple CAPM / Fama‑French 3‑factor Regression for **one stock** (Apple)
======================================================================
This minimal script shows how to:
1. Load the **APPL.xlsx** file you uploaded (daily prices).
2. Convert to **monthly** frequency and compute simple returns.
3. Download monthly Fama‑French 3 factors (Mkt‑RF, SMB, HML, RF).
4. Compute **excess returns** for Apple:  R_excess = R_AAPL − RF.
5. Run an OLS regression  R_excess = α + β_MKT·(Mkt‑RF) + β_SMB·SMB + β_HML·HML + ε.
6. Print the regression summary table.

Why no Fama–MacBeth here?  That method needs **many stocks** each month to form a
cross‑section.  With just Apple we can only do the **time‑series** step.  Once you
have more tickers you can extend this into the two‑pass FM procedure.
"""

from __future__ import annotations
import pandas as pd
import statsmodels.api as sm
import numpy as np

# ------------------------------------------------------------
# 1. Load Apple prices (daily) from the uploaded Excel file
# ------------------------------------------------------------
PRICE_FILE = "APPL.xlsx"  # adjust path if needed

aapl_daily = pd.read_excel(PRICE_FILE, parse_dates=["Date"])
aapl_daily.set_index("Date", inplace=True)

# Use adjusted close if available
price_col = "Adj Close" if "Adj Close" in aapl_daily.columns else "Close"

# 2. Resample to month‑end and compute returns
# ------------------------------------------------------------
aapl_monthly_price = aapl_daily[price_col].resample("M").last()
aapl_rets = aapl_monthly_price.pct_change().dropna()
aapl_rets.index = aapl_rets.index.to_period("M")

# ------------------------------------------------------------
# 3. Download Fama‑French 3 factors (monthly)
# ------------------------------------------------------------
FACTORS_URL = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
    "F-F_Research_Data_Factors.CSV"
)
ff = pd.read_csv(FACTORS_URL, skiprows=3)
ff = ff.rename(columns={ff.columns[0]: "Date"})

# Convert Date column to string first before filtering
ff["Date"] = ff["Date"].astype(str)

# Filter out rows that don't contain numeric date values
# This will remove rows with text like "Annual Factors: January-December"
ff = ff[ff["Date"].str.match(r'^\d+$')]  # Only keep rows where Date contains only digits

# Now convert to datetime after filtering out problematic rows
ff["Date"] = pd.to_datetime(ff["Date"] + "01", format="%Y%m%d")
ff.set_index("Date", inplace=True)
ff = ff.astype(float) / 100.0  # convert % → decimal
ff.index = ff.index.to_period("M")
ff = ff[["Mkt-RF", "SMB", "HML", "RF"]]


# ------------------------------------------------------------
# 4. Merge Apple returns with factors & compute excess returns
# ------------------------------------------------------------
common_idx = aapl_rets.index.intersection(ff.index)
aapl_rets = aapl_rets.loc[common_idx]
ff = ff.loc[common_idx]

aapl_excess = aapl_rets - ff["RF"]

# ------------------------------------------------------------
# 5. Run OLS regression (FF3)
# ------------------------------------------------------------
X = ff[["Mkt-RF", "SMB", "HML"]]
X = sm.add_constant(X)
model = sm.OLS(aapl_excess.values, X.values)
res = model.fit()

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------
print("=== Apple vs. Fama-French 3 Factors ===")
print(res.summary(xname=["α", "β_MKT", "β_SMB", "β_HML"]))

# Optional: save residuals / fitted values
resid = pd.Series(res.resid.flatten(), index=common_idx, name="resid")
resid.to_csv("aapl_ff3_residuals.csv")

C:\Users\tao11\AppData\Local\Temp\ipykernel_54648\1890801355.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  aapl_monthly_price = aapl_daily[price_col].resample("M").last()


=== Apple vs. Fama-French 3 Factors ===
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.739
Model:                            OLS   Adj. R-squared:                  0.710
Method:                 Least Squares   F-statistic:                     25.48
Date:                Fri, 20 Jun 2025   Prob (F-statistic):           4.94e-08
Time:                        22:58:23   Log-Likelihood:                 50.052
No. Observations:                  31   AIC:                            -92.10
Df Residuals:                      27   BIC:                            -86.37
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
α           